In [1]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import pandas as pd
import re
import os

In [2]:
# =========================
# 2. LOAD DATA
# =========================
df = pd.read_csv("../data/raw/customer-service-dpo.csv")

# show first rows
df.head()

,prompt,chosen,rejected
0,"I would like to acivate a card, can you help me?",To activate your Empowerbank card you have to ...,"I would like to acivate a card, can you help m..."
1,"I have to activate an Visa online, how can I d...","We apologize, as of now Empowerbank does not h...","I have to activate an Visa online, how can I d..."
2,"I would like to acivate a card, can you help me?",To activate your Empowerbank card you have to ...,"I would like to acivate a card, can you help m..."
3,"I have to activate an Visa online, how can I d...","We apologize, as of now Empowerbank does not h...","I have to activate an Visa online, how can I d..."
4,"I got to take out a loan, how to do it?","To take out a loan at Empowerbank, the first s...","I got to take out a loan, how to do it? * You..."


In [3]:
# =========================
# 3. BASIC CLEANING
# =========================

# remove empty rows
df = df.dropna()

# remove duplicate rows
df = df.drop_duplicates()

In [4]:
# =========================
# 4. TEXT CLEANING FUNCTION
# =========================

def clean_text(text):
    # convert to string
    text = str(text)
    
    # lower case (standardize text)
    text = text.lower()
    
    # remove special characters
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    
    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [6]:
print(df.columns)

Index(['prompt', 'chosen', 'rejected'], dtype='str')


In [7]:
df["prompt"] = df["prompt"].apply(clean_text)
df["chosen"] = df["chosen"].apply(clean_text)

In [8]:
df["text"] = "Question: " + df["prompt"] + "\nAnswer: " + df["chosen"]

In [9]:
df = df[["text"]]

In [10]:
# =========================
# 7. REMOVE VERY SHORT TEXT
# =========================

# keep only useful data (avoid noise)
df = df[df["text"].str.len() > 20]

In [11]:
# =========================
# 8. RESET INDEX
# =========================

df = df.reset_index(drop=True)

In [12]:
import os

output_dir = "../data/clean_data"
os.makedirs(output_dir, exist_ok=True)

for i, row in df.iterrows():
    file_path = os.path.join(output_dir, f"doc_{i}.txt")
    
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(row["text"])

print("✅ Clean RAG TXT files saved")

✅ Clean RAG TXT files saved


In [13]:
import os

folder_path = "../data/clean_data"

files = os.listdir(folder_path)

documents = []

for file in files:
    with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:
        text = f.read()
        documents.append(text)

print(len(documents))

5732


In [38]:
# =========================
# CHECK TOP 10 CHUNKS
# =========================

print(f"Total chunks: {len(chunks)}")

print("\n====================")
print("TOP 10 CHUNKS PREVIEW")
print("====================\n")

for i, chunk in enumerate(chunks[:10]):
    print(f"====================")
    print(f"CHUNK {i+1}")
    print(f"====================\n")
    
    print(chunk)
    
    print("\nLength:", len(chunk))
    
    # simple structure check
    if "Question" in chunk:
        print("✔ Has Question")
    if "Answer" in chunk:
        print("✔ Has Answer")

    print("\n")

Total chunks: 1

TOP 10 CHUNKS PREVIEW

CHUNK 1

Question: i would like to acivate a card can you help me
Answer: to activate your empowerbank card you have to make an initial deposit of us5 or equivalent that is if it is first time opening an account in case that you have lost your card you need not to deposit anything but just buying a new card from the closest branch and they will link it to your account

Length: 361
✔ Has Question
✔ Has Answer




In [39]:
# =========================
# CHUNK STATISTICS
# =========================

sizes = [len(c) for c in chunks]

print("Min size:", min(sizes))
print("Max size:", max(sizes))
print("Avg size:", sum(sizes)/len(sizes))

# classification
small = sum(1 for s in sizes if s < 200)
medium = sum(1 for s in sizes if 200 <= s <= 600)
large = sum(1 for s in sizes if s > 600)

print("\nSmall chunks:", small)
print("Medium chunks:", medium)
print("Large chunks:", large)

Min size: 361
Max size: 361
Avg size: 361.0

Small chunks: 0
Medium chunks: 1
Large chunks: 0


In [40]:
import pandas as pd

df = pd.DataFrame({
    "chunk": chunks[:10],
    "length": [len(c) for c in chunks[:10]]
})

df

,chunk,length
0,Question: i would like to acivate a card can y...,361


In [49]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter

folder_path = "../data/clean_data"  # adjust if needed

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=100
)

all_chunks = []

files = [f for f in os.listdir(folder_path) if f.endswith(".txt")]

print("Total files found:", len(files))

for file in files:
    file_path = os.path.join(folder_path, file)

    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read().strip()

        if not text:
            continue

        chunks = splitter.split_text(text)
        all_chunks.extend(chunks)

print("\nTotal chunks:", len(all_chunks))

Total files found: 5732

Total chunks: 7586


In [50]:
print("First file:", files[0])
print("Sample chunk:\n", all_chunks[0])

First file: doc_0.txt
Sample chunk:
 Question: i would like to acivate a card can you help me
Answer: to activate your empowerbank card you have to make an initial deposit of us5 or equivalent that is if it is first time opening an account in case that you have lost your card you need not to deposit anything but just buying a new card from the closest branch and they will link it to your account


In [51]:
# =========================
# TOP 10 CHUNKS INSPECTION
# =========================

print("Total chunks:", len(all_chunks))

print("\n==============================")
print("TOP 10 CHUNKS PREVIEW")
print("==============================\n")

for i, chunk in enumerate(all_chunks[:10]):
    print("=================================")
    print(f"CHUNK {i+1}")
    print("=================================\n")
    
    print(chunk)
    
    print("\nLength:", len(chunk))
    
    # structure check
    if "Question" in chunk:
        print("✔ Has Question")
    if "Answer" in chunk:
        print("✔ Has Answer")
    
    print("\n")

Total chunks: 7586

TOP 10 CHUNKS PREVIEW

CHUNK 1

Question: i would like to acivate a card can you help me
Answer: to activate your empowerbank card you have to make an initial deposit of us5 or equivalent that is if it is first time opening an account in case that you have lost your card you need not to deposit anything but just buying a new card from the closest branch and they will link it to your account

Length: 361
✔ Has Question
✔ Has Answer


CHUNK 2

Question: i have to activate an visa online how can i do it
Answer: we apologize as of now empowerbank does not have online visa capabilities you can try a physical card for your transactions and payments

Length: 204
✔ Has Question
✔ Has Answer


CHUNK 3

Question: im looking for a loan
Answer: if you are looking for a loan empowerbank has several options available including ssb loans consumer loans joint liability group loans micro entrepreneurbusiness loans and sme loans you can find the requirements for each of these in the 

In [52]:
# =========================
# CHUNK SIZE ANALYSIS
# =========================

sizes = [len(c) for c in all_chunks]

print("Min size:", min(sizes))
print("Max size:", max(sizes))
print("Avg size:", sum(sizes) / len(sizes))

# distribution check
small = sum(1 for s in sizes if s < 200)
medium = sum(1 for s in sizes if 200 <= s <= 600)
large = sum(1 for s in sizes if s > 600)

print("\nSmall chunks (<200):", small)
print("Medium chunks (200–600):", medium)
print("Large chunks (>600):", large)

Min size: 22
Max size: 400
Avg size: 211.155286053256

Small chunks (<200): 3546
Medium chunks (200–600): 4040
Large chunks (>600): 0
